# **Custom CNN Model**



> Necessary libraries are imported



In [1]:
import pandas as pd
import numpy as np
import os, re
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix
import seaborn as sns
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt


> Connecting to Google Drive (ONLY if the file is being run on Google Collab)




In [2]:
from google.colab import drive

#Mounting Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


> Configure paths and constants before running any cells below.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Adjust these paths before running.
# When using Google Colab set DATA_ROOT inside your mounted Drive.

DATA_ROOT       = "/content/drive/MyDrive/Hanan Share/Data_augmentation_updated"
MODEL_SAVE_PATH = "/content/drive/MyDrive"  # directory for saved model files

IMG_SIZE    = 224    # height and width of each input image (pixels)
NORM_FACTOR = 40.0   # max LLS height value used for [0,1] normalisation
NUM_CLASSES = 4      # Infill=0, Solid=1, Wire Good=2, Wire Bad=3
TEST_SPLIT  = 0.2    # fraction of dataset held out for testing
RANDOM_SEED = 42


# **Loading Augmented Data**

In [ ]:
def load_class(class_subdir):
    """Return {filename_stem: DataFrame} for all CSVs in DATA_ROOT/class_subdir."""
    class_dir = os.path.join(DATA_ROOT, class_subdir)
    files = [f for f in os.listdir(class_dir) if f.endswith('.csv')]
    return {
        os.path.splitext(f)[0]: pd.read_csv(os.path.join(class_dir, f), header=None)
        for f in files
    }


In [ ]:
Infill_df    = load_class("Infill")
Solid_df     = load_class("Solid")
Wire_good_df = load_class("Wire_good")
Wire_bad_df  = load_class("Wire_bad")

print(f"Infill: {len(Infill_df)}, Solid: {len(Solid_df)}, "
      f"Wire Good: {len(Wire_good_df)}, Wire Bad: {len(Wire_bad_df)}")


# **Concatenating all Data Frames for all Classes and Assigning Labels**

In [ ]:
concatenate_data_dic = {**Infill_df, **Solid_df, **Wire_good_df, **Wire_bad_df}

# Trim any off-by-one extra row/col produced during augmentation
for key in concatenate_data_dic:
    df = concatenate_data_dic[key]
    if df.shape[0] == IMG_SIZE + 1:
        concatenate_data_dic[key] = df.iloc[:-1, :]
    if df.shape[1] == IMG_SIZE + 1:
        concatenate_data_dic[key] = concatenate_data_dic[key].iloc[:, :-1]

data_values = {k: v.values for k, v in concatenate_data_dic.items()}

# Numeric labels: 0=Infill, 1=Solid, 2=Wire Good, 3=Wire Bad
labels = np.array(
    [0] * len(Infill_df) +
    [1] * len(Solid_df) +
    [2] * len(Wire_good_df) +
    [3] * len(Wire_bad_df)
)


# **Reshaping and Normalization of Data**

In [ ]:
for key in data_values:
    data_values[key] = data_values[key].reshape(IMG_SIZE, IMG_SIZE, 1) / NORM_FACTOR

data = np.array(list(data_values.values()))
print(f"Dataset shape: {data.shape}")


# **Splitting into Training and Testing Sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=TEST_SPLIT, random_state=RANDOM_SEED
)


# **Custom CNN Model**

In [ ]:
model = Sequential([
    layers.Conv2D(32, (3, 3), activation='PReLU', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Conv2D(64, (3, 3), activation='PReLU'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Conv2D(64, (3, 3), activation='PReLU'),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(64, activation='PReLU'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_split=TEST_SPLIT,
    callbacks=[early_stopping],
)


# **Evaluation of the Custom CNN Model**

In [ ]:
model.evaluate(X_test, y_test)


In [ ]:
y_pred_classes = np.argmax(model.predict(X_test), axis=1)


# **Classification Report and Confusion Matrix**

In [ ]:
CLASS_NAMES = ['Infill', 'Solid', 'Wire_Good', 'Wire_Bad']


In [ ]:
print(classification_report(y_test, y_pred_classes, target_names=CLASS_NAMES))

conf_matrix = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix - Custom CNN')
plt.show()


# **Loss and Accuracy Plots**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

ax1.plot(history.history['loss'],     label='Training Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('Training and Validation Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()

ax2.plot(history.history['accuracy'],     label='Training Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()

plt.tight_layout()
plt.show()


# **Save Model**

In [ ]:
save_path = os.path.join(MODEL_SAVE_PATH, 'customCNN.keras')
model.save(save_path)
print(f'Model saved to {save_path}')


# **Custom CNN Architecture**

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True)
img = plt.imread('model_plot.png')
plt.figure(figsize=(10, 10))
plt.imshow(img); plt.axis('off')
plt.show()
